In [40]:
import os
import re

import nltk
import numpy as np
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [22]:
from lda import get_nltk_data

In [23]:
get_nltk_data()

[nltk_data] Downloading package stopwords to
[nltk_data]     /data/users/kevinge/ML4NLP/a3/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     /data/users/kevinge/ML4NLP/a3/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /data/users/kevinge/ML4NLP/a3/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [100]:
class Preprocess:

    def __init__(self, document_file, min_frequency=10): # remove documents when not needed

        self.document_file = document_file
        self.min_frequency = min_frequency

        self.documents = self._load_documents()
        self.tokenized_documents = self._tokenize_documents()
        self.vocabulary = self._create_vocabulary()
        self.word_to_index = self._map_vocabulary_to_indices()
        self.filtered_documents = self._filter_tokenized_documents()
        self.documents_indices = self._convert_documents_to_indices()

    # load documents from the file and extract text within <TEXT> tags
    def _load_documents(self):
        with open(self.document_file, 'r') as f:
            content = f.read()
        documents = [t.strip() for t in re.findall(r"<TEXT>(.*?)</TEXT>", content, re.DOTALL)]
        print(f"Documents loaded:       {len(documents):,}")

        return documents

    # go through each document and tokenize it (convert to lowercase, remove punctuation and hyphened words, stopwords)
    def _tokenize_documents(self):
        tokenized_documents = []
        stop_words = set(stopwords.words('english'))
        stop_words |= {
            "said", "say", "told", "news",
            "would", "could", "may",
            "one", "two", "three", "first", "last",
            "year", "years", "day", "week", "today", "time", "since", "new",
            "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday",
            "also", "many", "made", "people",
        }
        for doc in self.documents:
            text = doc.lower()
            tokens = [
                token for token in word_tokenize(text, language="english")
                if token.isalpha()            # words only
                and len(token) > 2
                and token not in stop_words   # remove stopwords
            ]
            tokenized_documents.append(tokens)

        n_tokens = sum(len(t) for t in tokenized_documents)
        n_unique = len({t for tokens in tokenized_documents for t in tokens})
        print(f"Tokens after cleaning:  {n_tokens:,}  ({n_unique:,} unique)")

        return tokenized_documents

    # create a vocabulary of all words in the tokenized documents and filter based on minimum frequency
    def _create_vocabulary(self):
        word_count = Counter(t for tokens in self.tokenized_documents for t in tokens)

        filtered_word_count = {word: count for word, count in word_count.items() if count >= self.min_frequency}
        vocabulary = sorted(filtered_word_count)

        print(f"Vocabulary size:        {len(word_count):,} -> {len(vocabulary):,} (min frequency {self.min_frequency})")
        print("Top 20 words:", ", ".join(w for w, _ in word_count.most_common(20)))

        return vocabulary
        
    # map vocabulary to indices
    def _map_vocabulary_to_indices(self):
        return {word: idx for idx, word in enumerate(self.vocabulary)}

    def _filter_tokenized_documents(self):
        return [
            [token for token in tokens if token in self.word_to_index]
            for tokens in self.tokenized_documents
        ]

    # convert documents to indices based on the filtered vocabulary
    def _convert_documents_to_indices(self):
        return [
            [self.word_to_index[token] for token in tokens]   # already filtered, so no check needed
            for tokens in self.filtered_documents
        ]

In [101]:
p = Preprocess(document_file="data/ap.txt", min_frequency=10)

Documents loaded:       2,246
Tokens after cleaning:  462,647  (32,929 unique)
Vocabulary size:        32,929 -> 7,102 (min frequency 10)
Top 20 words: percent, million, president, government, state, bush, police, united, soviet, officials, states, billion, national, american, company, house, federal, court, city, party


In [52]:
p.documents[:2]

["A 16-year-old student at a private Baptist school who allegedly killed one teacher and wounded another before firing into a filled classroom apparently ``just snapped,'' the school's pastor said. ``I don't know how it could have happened,'' said George Sweet, pastor of Atlantic Shores Baptist Church. ``This is a good, Christian school. We pride ourselves on discipline. Our kids are good kids.'' The Atlantic Shores Christian School sophomore was arrested and charged with first-degree murder, attempted murder, malicious assault and related felony charges for the Friday morning shooting. Police would not release the boy's name because he is a juvenile, but neighbors and relatives identified him as Nicholas Elliott. Police said the student was tackled by a teacher and other students when his semiautomatic pistol jammed as he fired on the classroom as the students cowered on the floor crying ``Jesus save us! God save us!'' Friends and family said the boy apparently was troubled by his gra

In [53]:
for tokens in p.tokenized_documents[:2]:
    print(tokens[:15])

['student', 'private', 'baptist', 'school', 'allegedly', 'killed', 'one', 'teacher', 'wounded', 'another', 'firing', 'filled', 'classroom', 'apparently', 'snapped']
['bechtel', 'group', 'offered', 'sell', 'oil', 'israel', 'discount', 'least', 'million', 'years', 'promised', 'bomb', 'proposed', 'iraqi', 'pipeline']


In [55]:
p.vocabulary[:10]

['aaron',
 'abandon',
 'abandoned',
 'abc',
 'ability',
 'ablaze',
 'able',
 'aboard',
 'abortion',
 'abortions']

In [59]:
print([element for i, element in enumerate(p.word_to_index.items()) if i < 10])

[('aaron', 0), ('abandon', 1), ('abandoned', 2), ('abc', 3), ('ability', 4), ('ablaze', 5), ('able', 6), ('aboard', 7), ('abortion', 8), ('abortions', 9)]


In [65]:
for doc in p.documents_indices[:2]:
    print(", ".join(map(str, doc)))

6176, 4905, 570, 5623, 211, 3511, 4388, 6368, 7079, 288, 2520, 2493, 1165, 313, 5923, 5623, 4570, 5562, 3531, 1499, 2915, 5562, 2728, 6304, 4570, 453, 570, 1129, 2774, 1121, 5623, 4883, 3509, 2774, 3509, 453, 1121, 5623, 387, 1067, 4168, 464, 4168, 422, 5250, 2460, 1068, 2641, 4124, 5800, 4759, 7077, 5259, 777, 4187, 4240, 5257, 3123, 4266, 4759, 5562, 6176, 6368, 6177, 4698, 3392, 2517, 1165, 6177, 2551, 3414, 5601, 2765, 5601, 2645, 2402, 5562, 777, 313, 6632, 2807, 1668, 1939, 4535, 4266, 5562, 5596, 777, 4535, 5711, 2615, 7103, 153, 2807, 7013, 1842, 3578, 7102, 3757, 3133, 228, 5562, 2878, 777, 236, 6341, 2878, 5562, 3526, 3775, 3526, 4189, 4288, 3727, 2878, 3694, 4309, 3694, 677, 4389, 5893, 6368, 3123, 3474, 7079, 6368, 5577, 5716, 1331, 5596, 2879, 7081, 5814, 4759, 5562, 777, 228, 5812, 6461, 6368, 6287, 214, 2538, 5506, 5812, 5812, 5540, 6461, 1165, 672, 1161, 3969, 5812, 2756, 3750, 1963, 4396, 2515, 4759, 6025, 3669, 5562, 7122, 4698, 3392, 6368, 3926, 6177, 5562, 5670, 698

In [67]:
class LDA:

    def __init__(self, documents, V, alpha, beta, k, n_iter = 100, seed=0):

        self.documents = documents
        self.D = len(documents)
        self.V = V

        self.alpha = alpha
        self.beta = beta
        self.k = k
        self.n_iter = n_iter

        # randomly initialize the topic assignments for each word in each document
        self.rng = np.random.default_rng(seed)
        self.z = [self.rng.choice(self.k, size=len(doc)) for doc in self.documents]
    
        self.n_dk = np.zeros((self.D, self.k), dtype=int)
        self.m_kv = np.zeros((self.k, self.V), dtype=int)
        self.m_k = np.zeros(self.k, dtype=int)

        self._init_counts()

    def _init_counts(self):
        # go through every token once and count its random starting label
        for d, doc in enumerate(self.documents):
            for j, v in enumerate(doc):
                self._add(d, v, self.z[d][j])

    def _add(self, d, v, k):
        # put one token (word v in doc d, topic k) into the counts
        self.n_dk[d, k] += 1
        self.m_kv[k, v] += 1
        self.m_k[k] += 1

    def _remove(self, d, v, k):
        self.n_dk[d, k] -= 1
        self.m_kv[k, v] -= 1
        self.m_k[k] -= 1

    def p_k(self, d, v):
        # probability of each topic for word v in doc d (current token already removed)
        q_k = (self.alpha + self.n_dk[d]) * (self.beta + self.m_kv[:, v]) / (self.beta * self.V + self.m_k)
        return q_k / q_k.sum()

    def collapsed_gibbs(self):
        for it in range(self.n_iter):              # one iteration = one pass over every token
            changed = 0
            for d, doc in enumerate(self.documents):
                for j, v in enumerate(doc):
                    old_k = self.z[d][j]
                    self._remove(d, v, old_k)                        # 1. take the label off
                    new_k = self.rng.choice(self.k, p=self.p_k(d, v))  # 2-3. score topics and sample
                    self._add(d, v, new_k)                           # 4. put it back
                    self.z[d][j] = new_k
                    changed += int(new_k != old_k)

            if (it + 1) % 10 == 0:
                print(f"iteration {it + 1}/{self.n_iter}: {changed:,} labels changed")

        return self.z

In [69]:
len(p.vocabulary)

7133

In [71]:
lda = LDA(
    documents=p.documents_indices,
    V = len(p.vocabulary),
    alpha = 0.1,
    beta = 0.1,
    k = 10
)

In [73]:
lda.z[:2]

[array([8, 6, 5, 2, 3, 0, 0, 0, 1, 8, 6, 9, 5, 6, 9, 7, 6, 5, 5, 9, 2, 8,
        6, 0, 3, 8, 5, 0, 7, 7, 8, 1, 0, 8, 0, 5, 0, 2, 4, 4, 4, 0, 0, 1,
        0, 6, 5, 6, 2, 6, 7, 3, 4, 9, 8, 9, 3, 6, 9, 6, 8, 6, 7, 3, 8, 1,
        5, 7, 8, 5, 3, 3, 4, 4, 7, 8, 0, 9, 5, 3, 6, 5, 2, 3, 7, 5, 5, 3,
        7, 3, 3, 8, 2, 2, 7, 6, 0, 0, 3, 8, 4, 7, 3, 2, 7, 8, 0, 0, 6, 3,
        5, 1, 8, 4, 8, 7, 7, 2, 7, 0, 5, 4, 9, 1, 9, 0, 6, 5, 8, 2, 9, 6,
        8, 1, 7, 9, 0, 3, 6, 1, 5, 6, 7, 9, 4, 4, 4, 9, 1, 4, 0, 4, 9, 6,
        3, 9, 6, 9, 0, 4, 8, 7, 4, 4, 4, 5, 2, 7, 0, 4, 2, 7, 7, 7, 9, 9,
        1, 1, 1, 7, 9, 9, 6, 9, 8, 0, 1, 8, 0, 9, 8, 9, 3, 1, 5, 9, 3, 8,
        3, 8, 2, 4, 3, 2, 8, 8, 1, 9, 9, 2, 4, 5, 6, 4, 1, 9, 6, 0, 8, 7,
        1, 6, 5, 0, 9, 7, 3, 0, 0, 7, 1, 5, 8, 9, 2, 0, 4, 8, 6, 0, 6, 3,
        2, 4, 8, 9, 1, 5, 7, 2, 2, 2, 2, 8, 2, 2, 1]),
 array([1, 7, 2, 8, 5, 8, 5, 7, 8, 0, 5, 4, 2, 4, 4, 4, 8, 8, 6, 7, 9, 6,
        3, 0, 5, 2, 5, 0, 8, 9, 1, 8, 4, 0, 9, 9, 0, 5, 8

In [74]:
z = lda.collapsed_gibbs()

iteration 10/100: 211,631 labels changed
iteration 20/100: 161,533 labels changed
iteration 30/100: 148,397 labels changed
iteration 40/100: 143,357 labels changed
iteration 50/100: 140,054 labels changed
iteration 60/100: 138,857 labels changed
iteration 70/100: 137,862 labels changed
iteration 80/100: 136,910 labels changed
iteration 90/100: 137,282 labels changed
iteration 100/100: 135,612 labels changed


In [75]:
phi = (lda.m_kv + lda.beta) / (lda.m_k[:, None] + lda.V * lda.beta)

In [76]:
phi

array([[6.64992574e-05, 1.61498197e-04, 9.81655705e-05, ...,
        6.64992574e-05, 3.16663131e-06, 3.16663131e-06],
       [2.02933196e-06, 6.29092909e-05, 2.02933196e-06, ...,
        2.04962528e-04, 2.02933196e-06, 2.02933196e-06],
       [3.08144569e-06, 1.26339273e-04, 3.08144569e-06, ...,
        3.08144569e-06, 1.87968187e-04, 3.38959026e-05],
       ...,
       [2.85973296e-06, 2.85973296e-06, 2.85973296e-06, ...,
        2.85973296e-06, 2.85973296e-06, 6.60598313e-04],
       [1.21602815e-04, 1.50126932e-06, 3.15266558e-05, ...,
        1.50126932e-06, 1.06590122e-04, 1.50126932e-06],
       [2.24611590e-06, 2.24611590e-06, 4.71684340e-05, ...,
        2.24611590e-06, 2.24611590e-06, 2.24611590e-06]], shape=(10, 7133))

In [77]:
doc_length = lda.n_dk.sum(axis=1, keepdims=True)
theta = (lda.n_dk + lda.alpha) / (doc_length + lda.k * lda.alpha)

In [78]:
theta

array([[3.87596899e-04, 3.87596899e-04, 3.87596899e-04, ...,
        3.87596899e-04, 4.73255814e-01, 9.34108527e-02],
       [3.26797386e-04, 3.76143791e-01, 1.40849673e-01, ...,
        3.26797386e-04, 4.93464052e-02, 3.26797386e-04],
       [4.18410042e-04, 4.18410042e-04, 4.60251046e-03, ...,
        4.18410042e-04, 5.65271967e-01, 8.82845188e-02],
       ...,
       [8.40336134e-04, 8.40336134e-04, 8.40336134e-04, ...,
        3.44537815e-02, 2.19327731e-01, 5.89075630e-01],
       [1.11111111e-03, 1.11111111e-03, 1.11111111e-03, ...,
        1.56666667e-01, 6.67777778e-01, 1.45555556e-01],
       [1.42857143e-03, 1.42857143e-03, 1.42857143e-03, ...,
        1.42857143e-03, 1.01428571e-01, 1.42857143e-03]], shape=(2246, 10))

In [79]:
n = 10
for k in range(lda.k):
    top = np.argsort(-phi[k])[:n]                  # word IDs, highest probability first
    print(f"Topic {k}: " + ", ".join(p.vocabulary[v] for v in top))

Topic 0: bush, said, dukakis, campaign, president, percent, democratic, jackson, new, republican
Topic 1: said, united, states, soviet, would, president, military, west, bush, east
Topic 2: said, million, company, billion, year, new, sales, percent, last, would
Topic 3: said, government, party, soviet, people, political, president, south, gorbachev, country
Topic 4: said, police, people, two, one, killed, miles, officials, fire, city
Topic 5: said, would, house, committee, bill, senate, federal, congress, office, department
Topic 6: said, percent, year, workers, would, report, new, million, years, program
Topic 7: percent, said, market, prices, dollar, new, rose, stock, higher, rates
Topic 8: said, one, years, new, people, like, first, school, time, get
Topic 9: said, court, case, judge, drug, trial, attorney, charges, state, prison


In [91]:
for i in np.argsort(lda.m_kv[1])[::-1][:10]:
    print(p.vocabulary[i])

said
united
states
soviet
would
president
military
west
bush
east


In [92]:
lda.m_kv

array([[  0,   0,   0, ..., 449, 789, 796],
       [  0,   3,   0, ...,  10,   0,   0],
       [  0,   4,   0, ...,   0,   6,   1],
       ...,
       [  0,   0,   0, ...,   0,   0,  23],
       [  8,   0,   2, ...,   0,   7,   0],
       [  0,   0,   2, ...,   0,   0,   0]], shape=(10, 7133))

In [95]:
def umass_coherence(phi, documents, V, M=20):
    # X[d, v] = 1 if word v appears at least once in document d
    X = np.zeros((len(documents), V), dtype=np.uint8)
    for d, doc in enumerate(documents):
        X[d, list(set(doc))] = 1

    scores = []
    for t in range(phi.shape[0]):
        top = np.argsort(-phi[t])[:M]            # top M word IDs, most probable first
        sub = X[:, top].astype(np.int32)         # D x M: which docs contain each top word
        co = sub.T @ sub                         # co[m, l] = D(v_m, v_l), co[l, l] = D(v_l)

        score = 0.0
        for m in range(1, M):
            for l in range(m):                   # l ranks above m
                score += np.log((co[m, l] + 1) / co[l, l])
        scores.append(score)

    return np.array(scores)

In [96]:
scores = umass_coherence(phi, p.documents_indices, lda.V, M=20)

for t in np.argsort(-scores):                    # most coherent first
    top = np.argsort(-phi[t])[:10]
    print(f"{scores[t]:8.1f}  Topic {t}: " + ", ".join(p.vocabulary[v] for v in top))

print("Mean coherence:", scores.mean())

  -241.5  Topic 9: said, court, case, judge, drug, trial, attorney, charges, state, prison
  -249.4  Topic 5: said, would, house, committee, bill, senate, federal, congress, office, department
  -259.7  Topic 2: said, million, company, billion, year, new, sales, percent, last, would
  -266.6  Topic 6: said, percent, year, workers, would, report, new, million, years, program
  -275.8  Topic 8: said, one, years, new, people, like, first, school, time, get
  -288.6  Topic 0: bush, said, dukakis, campaign, president, percent, democratic, jackson, new, republican
  -288.8  Topic 7: percent, said, market, prices, dollar, new, rose, stock, higher, rates
  -289.5  Topic 4: said, police, people, two, one, killed, miles, officials, fire, city
  -302.2  Topic 3: said, government, party, soviet, people, political, president, south, gorbachev, country
  -350.8  Topic 1: said, united, states, soviet, would, president, military, west, bush, east
Mean coherence: -281.2902971598411
